# Sign Bridge · WLASL video training

This private Kaggle notebook runs the video-to-landmark-to-model pipeline on the attached WLASL-processed dataset. It samples each usable clip at 64 frames, runs the same MediaPipe Holistic task asset bundled with Sign Bridge, selects the app's 180 landmark features, and trains a softmax classifier from raw videos.

The manifest keeps the WLASL official split and signer IDs. Only glosses with available examples in train, validation, and test can be evaluated; the 2,000-class annotations do **not** imply 2,000 usable classes in the local video copy. This produces a new classifier output contract `[1,64,180,3] → [1,N]`, so the mobile inference decoder will need an update before this model is installed in the app.

WLASL videos and derived features are academic/computational research data. Follow the WLASL C-UDA and cite Li et al., WACV 2020. This kernel is private; it does not upload the raw videos anywhere.

## Resume notes

Feature extraction writes each completed chunk atomically under `/kaggle/working/sign_bridge_wlasl_run/features/`; it skips valid chunks on a rerun. Training saves a TensorFlow model+optimizer checkpoint after every epoch. If the Kaggle session retains or reattaches `/kaggle/working`, the next run resumes from those checkpoints. A completed Kaggle notebook version saves its output files and they can be attached to a later version. Kaggle background `Save & Run All` sessions use a fresh environment; a forcibly cancelled background run is not guaranteed to preserve unsaved working files. For a durable restart, save/attach the latest completed output version before restarting. Do not delete the checkpoint files.

The six-gloss GPU preflight passed, including video decode, landmark extraction, one training epoch, and TFLite export. This version runs the full eligible WLASL-2000 dataset and logs progress at each extraction chunk and epoch.


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

# Full run on all eligible WLASL-2000 examples.
RUN_MODE = "full"

if importlib.util.find_spec("mediapipe") is None:
    # Kaggle's Python 3.13 runtime publishes MediaPipe 0.10.35+ wheels.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mediapipe==0.10.35"])

import mediapipe as mp
print("MediaPipe", mp.__version__)
TRAINER_SOURCE = "#!/usr/bin/env python3\n\"\"\"Extract Android-compatible Holistic sequences from WLASL MP4s and train a classifier.\n\nThe WLASL source videos are inputs only. Per-video feature chunks and training\ncheckpoints are written under /kaggle/working so completed chunks/epochs can be\nreused when Kaggle preserves or reattaches the prior run's outputs.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nimport cv2\nimport mediapipe as mp\nimport numpy as np\n\nFRAMES = 64\nLANDMARKS = 180\nCOORDS = 3\nNSLT_SUBSET = 2000\nMIN_TRAIN_PER_CLASS = 1\nTASK_FILENAME = \"holistic_landmarker.task\"\nFACE_INDICES = [\n    0, 4, 7, 8, 10, 13, 14, 17, 21, 33, 37, 39, 40, 46, 52, 53, 54, 55,\n    58, 61, 63, 65, 66, 67, 70, 78, 80, 81, 82, 84, 87, 88, 91, 93, 95,\n    103, 105, 107, 109, 127, 132, 133, 136, 144, 145, 146, 148, 149, 150,\n    152, 153, 154, 155, 157, 158, 159, 160, 161, 162, 163, 172, 173, 176,\n    178, 181, 185, 191, 234, 246, 249, 251, 263, 267, 269, 270, 276, 282,\n    283, 284, 285, 288, 291, 293, 295, 296, 297, 300, 308, 310, 311, 312,\n    314, 317, 318, 321, 323, 324, 332, 334, 336, 338, 356, 361, 362, 365,\n    373, 374, 375, 377, 378, 379, 380, 381, 382, 384, 385, 386, 387, 388,\n    389, 390, 397, 398, 400, 402, 405, 409, 415, 454, 466, 468, 473,\n]\nPOSE_INDICES = [11, 12, 13, 14, 15, 16]\n\n\ndef args_parser() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--mode\", choices=(\"preflight\", \"full\"), default=\"full\")\n    parser.add_argument(\"--data-root\", type=Path, default=Path(\"/kaggle/input/wlasl-processed\"))\n    parser.add_argument(\"--task-model\", type=Path, default=Path(\"/kaggle/input/sign-bridge-holistic-task-model/holistic_landmarker.task\"))\n    parser.add_argument(\"--work-dir\", type=Path, default=Path(\"/kaggle/working/sign_bridge_wlasl_run\"))\n    parser.add_argument(\"--epochs\", type=int, default=60)\n    parser.add_argument(\"--batch-size\", type=int, default=32)\n    parser.add_argument(\"--extract-chunk-size\", type=int, default=64)\n    return parser.parse_args()\n\n\ndef read_records(root: Path) -> tuple[list[dict[str, Any]], list[str], dict[str, Any]]:\n    classes = {}\n    for line in (root / \"wlasl_class_list.txt\").read_text(encoding=\"utf-8\").splitlines():\n        class_id, gloss = line.strip().split(maxsplit=1)\n        classes[int(class_id)] = gloss\n\n    with (root / f\"nslt_{NSLT_SUBSET}.json\").open(encoding=\"utf-8\") as source:\n        nslt = json.load(source)\n    with (root / \"WLASL_v0.3.json\").open(encoding=\"utf-8\") as source:\n        wlasl = json.load(source)\n\n    annotations: dict[str, dict[str, Any]] = {}\n    for word in wlasl:\n        for instance in word.get(\"instances\", []):\n            video_id = str(instance.get(\"video_id\", \"\"))\n            if video_id:\n                annotations[video_id] = {**instance, \"gloss\": str(word.get(\"gloss\", \"\"))}\n\n    video_root = root / \"videos\"\n    split_counts: dict[str, Counter[str]] = {s: Counter() for s in (\"train\", \"val\", \"test\")}\n    rows: list[dict[str, Any]] = []\n    missing_video = 0\n    missing_metadata = 0\n    for video_id, item in nslt.items():\n        video_id = str(video_id)\n        action = item.get(\"action\", [])\n        split = str(item.get(\"subset\", \"\")).lower()\n        if len(action) < 1 or split not in split_counts or action[0] not in classes:\n            continue\n        gloss = classes[action[0]]\n        if not (video_root / f\"{video_id}.mp4\").is_file():\n            missing_video += 1\n            continue\n        annotation = annotations.get(video_id)\n        if annotation is None or annotation.get(\"gloss\") != gloss:\n            missing_metadata += 1\n            continue\n        start = int(annotation.get(\"frame_start\", 1) or 1)\n        end = int(annotation.get(\"frame_end\", -1) or -1)\n        rows.append({\n            \"video_id\": video_id,\n            \"gloss\": gloss,\n            \"split\": split,\n            \"signer_id\": str(annotation.get(\"signer_id\", \"\")),\n            \"frame_start\": start,\n            \"frame_end\": end,\n            \"fps\": float(annotation.get(\"fps\", 25) or 25),\n            \"path\": str(video_root / f\"{video_id}.mp4\"),\n        })\n        split_counts[split][gloss] += 1\n\n    common = set.intersection(*(set(split_counts[s]) for s in (\"train\", \"val\", \"test\")))\n    eligible = {\n        gloss for gloss in common\n        if split_counts[\"train\"][gloss] >= MIN_TRAIN_PER_CLASS\n    }\n    rows = [row for row in rows if row[\"gloss\"] in eligible]\n    rows.sort(key=lambda row: (row[\"split\"], row[\"gloss\"], row[\"video_id\"]))\n    class_names = sorted(eligible)\n    summary = {\n        \"subset\": f\"WLASL{NSLT_SUBSET}\",\n        \"source_video_count\": len({row[\"video_id\"] for row in rows}),\n        \"usable_class_count\": len(class_names),\n        \"class_names\": class_names,\n        \"eligible_examples_by_split\": dict(Counter(row[\"split\"] for row in rows)),\n        \"source_videos_missing_locally\": missing_video,\n        \"annotation_join_mismatches\": missing_metadata,\n        \"eligibility\": f\"At least {MIN_TRAIN_PER_CLASS} available train clip and at least one available val and test clip per class.\",\n        \"split_policy\": \"Official NSLT train/val/test split; signer IDs are preserved and overlap must be audited.\",\n    }\n    return rows, class_names, summary\n\n\ndef make_preflight_rows(rows: list[dict[str, Any]], class_names: list[str]) -> tuple[list[dict[str, Any]], list[str]]:\n    # A deterministic small sample with all three splits for a few glosses.\n    selected_names = class_names[:6]\n    chosen = []\n    for gloss in selected_names:\n        for split in (\"train\", \"val\", \"test\"):\n            candidates = [row for row in rows if row[\"gloss\"] == gloss and row[\"split\"] == split]\n            chosen.extend(candidates[:2 if split != \"train\" else 3])\n    return chosen, selected_names\n\n\ndef extract_one(row: dict[str, Any], landmarker: Any) -> tuple[np.ndarray, float]:\n    cap = cv2.VideoCapture(row[\"path\"])\n    if not cap.isOpened():\n        raise ValueError(\"video_open_failed\")\n    fps = cap.get(cv2.CAP_PROP_FPS)\n    if not np.isfinite(fps) or fps < 1:\n        fps = 25.0\n    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))\n    if total < 1:\n        cap.release()\n        raise ValueError(\"video_has_no_frames\")\n    # WLASL boundaries are indexed at 25 fps; convert to the decoded file FPS.\n    start = max(0, int(round((row[\"frame_start\"] - 1) * fps / max(row[\"fps\"], 1.0))))\n    end = total - 1 if row[\"frame_end\"] < 0 else int(round((row[\"frame_end\"] - 1) * fps / max(row[\"fps\"], 1.0)))\n    start = min(start, total - 1)\n    end = min(max(end, start), total - 1)\n    targets = np.rint(np.linspace(start, end, FRAMES)).astype(np.int64)\n    target_slots: dict[int, list[int]] = defaultdict(list)\n    for slot, frame_index in enumerate(targets):\n        target_slots[int(frame_index)].append(slot)\n\n    sequence = np.zeros((FRAMES, LANDMARKS, COORDS), dtype=np.float32)\n    seen_hands = 0\n    cap.set(cv2.CAP_PROP_POS_FRAMES, start)\n    current_index = start\n    while current_index <= end:\n        ok, bgr = cap.read()\n        if not ok:\n            break\n        slots = target_slots.get(current_index)\n        if slots:\n            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)\n            # Holistic's temporal smoothing graph expects a stable image size\n            # across calls made by the same landmarker instance.\n            rgb = cv2.resize(rgb, (256, 256), interpolation=cv2.INTER_AREA)\n            result = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))\n            right = result.right_hand_landmarks or []\n            left = result.left_hand_landmarks or []\n            pose = result.pose_landmarks or []\n            face = result.face_landmarks or []\n            if right or left:\n                seen_hands += len(slots)\n            feature = np.zeros((LANDMARKS, COORDS), dtype=np.float32)\n            offset = 0\n            for group, indices in ((right, range(21)), (left, range(21)), (pose, POSE_INDICES), (face, FACE_INDICES)):\n                for index in indices:\n                    if group and index < len(group):\n                        lm = group[index]\n                        feature[offset] = (lm.x, lm.y, lm.z)\n                    offset += 1\n            for slot in slots:\n                sequence[slot] = feature\n        current_index += 1\n    cap.release()\n    if not np.any(sequence):\n        raise ValueError(\"no_landmarks_detected\")\n    return normalize_sequence(sequence), float(seen_hands / FRAMES)\n\n\ndef normalize_sequence(sequence: np.ndarray) -> np.ndarray:\n    seq = np.asarray(sequence, dtype=np.float32).copy()\n    if seq.shape != (FRAMES, LANDMARKS, COORDS) or not np.isfinite(seq).all():\n        raise ValueError(f\"invalid_sequence_shape_or_values: {seq.shape}\")\n    for frame in seq:\n        for start, end in ((0, 21), (21, 42)):\n            hand = frame[start:end]\n            if np.any(hand != 0):\n                hand -= hand[0].copy()\n                hand /= max(float(np.max(np.abs(hand))), 1e-6)\n    return seq\n\n\ndef locate_completed_chunks(work: Path, split: str) -> set[int]:\n    directory = work / \"features\" / split\n    return {int(path.stem.split(\"_\")[-1]) for path in directory.glob(\"chunk_*.npz\")}\n\n\ndef extract_chunks(rows: list[dict[str, Any]], class_names: list[str], task_model: Path,\n                   work: Path, chunk_size: int, mode: str) -> None:\n    class_to_id = {name: index for index, name in enumerate(class_names)}\n    task = mp.tasks.vision\n    landmarker = None\n    for delegate in (mp.tasks.BaseOptions.Delegate.GPU, mp.tasks.BaseOptions.Delegate.CPU):\n        options = task.HolisticLandmarkerOptions(\n            base_options=mp.tasks.BaseOptions(model_asset_path=str(task_model), delegate=delegate),\n            running_mode=task.RunningMode.IMAGE,\n            min_face_detection_confidence=0.5,\n            min_pose_detection_confidence=0.5,\n            min_hand_landmarks_confidence=0.5,\n        )\n        try:\n            landmarker = task.HolisticLandmarker.create_from_options(options)\n            print(f\"[extract] MediaPipe delegate: {delegate.name}\", flush=True)\n            break\n        except Exception as error:\n            print(f\"[extract] {delegate.name} delegate unavailable: {error}; trying fallback\", flush=True)\n    if landmarker is None:\n        raise RuntimeError(\"Could not initialize MediaPipe Holistic with GPU or CPU delegate\")\n    failures_path = work / \"extraction_failures.csv\"\n    progress_path = work / \"extraction_progress.json\"\n    chunk_total = 0\n    failure_counts: Counter[str] = Counter()\n    with landmarker:\n        for split in (\"train\", \"val\", \"test\"):\n            subset = [row for row in rows if row[\"split\"] == split]\n            if mode == \"preflight\":\n                subset = subset[:96]\n            out_dir = work / \"features\" / split\n            out_dir.mkdir(parents=True, exist_ok=True)\n            chunk_count = (len(subset) + chunk_size - 1) // chunk_size\n            for chunk_index in range(chunk_count):\n                target = out_dir / f\"chunk_{chunk_index:05d}.npz\"\n                start = chunk_index * chunk_size\n                chunk_rows = subset[start:start + chunk_size]\n                expected_ids = [row[\"video_id\"] for row in chunk_rows]\n                if target.is_file():\n                    try:\n                        with np.load(target, allow_pickle=False) as saved:\n                            if saved[\"video_ids\"].astype(str).tolist() == expected_ids:\n                                chunk_total += 1\n                                continue\n                    except Exception:\n                        target.unlink(missing_ok=True)\n                xs: list[np.ndarray] = []\n                ys: list[int] = []\n                ids: list[str] = []\n                ratios: list[float] = []\n                reasons: list[str] = []\n                for row in chunk_rows:\n                    try:\n                        x, hand_ratio = extract_one(row, landmarker)\n                        xs.append(x)\n                        ys.append(class_to_id[row[\"gloss\"]])\n                        ids.append(row[\"video_id\"])\n                        ratios.append(hand_ratio)\n                    except Exception as error:\n                        reasons.append(f\"{row['video_id']},{row['gloss']},{split},{type(error).__name__}:{error}\")\n                        failure_counts[type(error).__name__] += 1\n                temp = target.with_suffix(\".tmp.npz\")\n                np.savez_compressed(\n                    temp,\n                    x=np.stack(xs) if xs else np.zeros((0, FRAMES, LANDMARKS, COORDS), dtype=np.float32),\n                    y=np.asarray(ys, dtype=np.int32),\n                    video_ids=np.asarray(ids, dtype=str),\n                    hand_frame_ratio=np.asarray(ratios, dtype=np.float32),\n                )\n                os.replace(temp, target)\n                if reasons:\n                    with failures_path.open(\"a\", encoding=\"utf-8\") as failures:\n                        if failures.tell() == 0:\n                            failures.write(\"video_id,gloss,split,reason\\n\")\n                        failures.write(\"\\n\".join(reasons) + \"\\n\")\n                chunk_total += 1\n                if chunk_total % 5 == 0 or chunk_total == 1:\n                    progress_path.write_text(json.dumps({\n                        \"completed_chunks\": chunk_total,\n                        \"split\": split,\n                        \"chunk\": chunk_index,\n                        \"failures_by_exception\": dict(failure_counts),\n                        \"updated_unix\": time.time(),\n                    }, indent=2))\n                    print(f\"[extract] {split}: chunk {chunk_index + 1}/{chunk_count}; completed chunks={chunk_total}; failures={sum(failure_counts.values())}\", flush=True)\n\n\ndef load_split(work: Path, split: str) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\n    paths = sorted((work / \"features\" / split).glob(\"chunk_*.npz\"))\n    if not paths:\n        raise RuntimeError(f\"No extracted chunks for {split}\")\n    xs, ys, ids, hand_ratios = [], [], [], []\n    for path in paths:\n        with np.load(path, allow_pickle=False) as data:\n            if len(data[\"x\"]):\n                xs.append(data[\"x\"])\n                ys.append(data[\"y\"])\n                ids.append(data[\"video_ids\"])\n                hand_ratios.append(data[\"hand_frame_ratio\"])\n    if not xs:\n        raise RuntimeError(f\"All videos failed extraction for {split}\")\n    return np.concatenate(xs), np.concatenate(ys), np.concatenate(ids), np.concatenate(hand_ratios)\n\n\ndef build_classifier(tf: Any, classes: int, unroll: bool = False) -> Any:\n    inputs = tf.keras.Input((FRAMES, LANDMARKS, COORDS), name=\"landmarks\")\n    x = tf.keras.layers.Reshape((FRAMES, LANDMARKS * COORDS), name=\"flatten_frame\")(inputs)\n    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(128, activation=\"relu\"), name=\"frame_projection\")(x)\n    x = tf.keras.layers.TimeDistributed(tf.keras.layers.LayerNormalization(), name=\"frame_normalization\")(x)\n    x = tf.keras.layers.Dropout(0.15)(x)\n    x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(128, return_sequences=True, unroll=unroll), name=\"bigru\")(x)\n    x = tf.keras.layers.LayerNormalization()(x)\n    scores = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(1), name=\"temporal_attention_scores\")(x)\n    weights = tf.keras.layers.Softmax(axis=1, name=\"temporal_attention_weights\")(scores)\n    x = tf.keras.layers.Multiply()([x, weights])\n    x = tf.keras.layers.Lambda(lambda value: tf.reduce_sum(value, axis=1), name=\"temporal_pool\")(x)\n    x = tf.keras.layers.Dense(256, activation=\"relu\")(x)\n    x = tf.keras.layers.Dropout(0.35)(x)\n    outputs = tf.keras.layers.Dense(classes, activation=\"softmax\", name=\"gloss_probabilities\")(x)\n    return tf.keras.Model(inputs, outputs, name=\"SignBridge_WLASL_video_classifier\")\n\n\ndef evaluate_model(model: Any, x: np.ndarray, y: np.ndarray, class_names: list[str],\n                   ids: np.ndarray, work: Path) -> dict[str, Any]:\n    from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support\n    probabilities = model.predict(x, batch_size=64, verbose=1)\n    top = np.argsort(probabilities, axis=1)[:, ::-1]\n    pred = top[:, 0]\n    per_class = precision_recall_fscore_support(y, pred, labels=np.arange(len(class_names)), zero_division=0)\n    metrics = {\n        \"samples\": int(len(y)),\n        \"labels\": len(class_names),\n        \"top1_accuracy\": float(accuracy_score(y, pred)),\n        \"top5_accuracy\": float(np.mean([int(target) in ranks[:5] for target, ranks in zip(y, top)])),\n        \"macro_precision\": float(np.mean(per_class[0])),\n        \"macro_recall\": float(np.mean(per_class[1])),\n        \"macro_f1\": float(f1_score(y, pred, labels=np.arange(len(class_names)), average=\"macro\", zero_division=0)),\n    }\n    with (work / \"per_class_metrics.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as file:\n        writer = csv.writer(file)\n        writer.writerow((\"gloss\", \"precision\", \"recall\", \"f1\", \"support\"))\n        for index, name in enumerate(class_names):\n            writer.writerow((name, per_class[0][index], per_class[1][index], per_class[2][index], per_class[3][index]))\n    matrix = confusion_matrix(y, pred, labels=np.arange(len(class_names)))\n    np.savetxt(work / \"confusion_matrix.csv\", matrix, delimiter=\",\", fmt=\"%d\")\n    np.savez_compressed(work / \"test_predictions.npz\", video_ids=ids, y_true=y, y_pred=pred, top5=top[:, :5])\n    return metrics\n\n\ndef train(args: argparse.Namespace, rows: list[dict[str, Any]], class_names: list[str],\n          work: Path, epochs: int) -> None:\n    import tensorflow as tf\n    if not tf.config.list_physical_devices(\"GPU\"):\n        raise RuntimeError(\"A Kaggle GPU was requested but TensorFlow sees no GPU; stopping before training.\")\n    x_train, y_train, train_ids, train_hand_ratio = load_split(work, \"train\")\n    x_val, y_val, val_ids, val_hand_ratio = load_split(work, \"val\")\n    x_test, y_test, test_ids, test_hand_ratio = load_split(work, \"test\")\n    if len(class_names) < 2 or min(len(y_train), len(y_val), len(y_test)) < 1:\n        raise RuntimeError(\"Dataset has insufficient usable classes/splits\")\n\n    model = build_classifier(tf, len(class_names))\n    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)\n    model.compile(optimizer=optimizer, loss=\"sparse_categorical_crossentropy\", metrics=[\"accuracy\"])\n    checkpoint_dir = work / \"checkpoints\"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    epoch_variable = tf.Variable(0, trainable=False, dtype=tf.int64, name=\"completed_epochs\")\n    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer, epoch=epoch_variable)\n    manager = tf.train.CheckpointManager(checkpoint, str(checkpoint_dir), max_to_keep=3)\n    initial_epoch = 0\n    if manager.latest_checkpoint:\n        checkpoint.restore(manager.latest_checkpoint).expect_partial()\n        initial_epoch = int(epoch_variable.numpy())\n        print(f\"[resume] restored {manager.latest_checkpoint}; next epoch={initial_epoch + 1}\", flush=True)\n\n    class CheckpointEveryEpoch(tf.keras.callbacks.Callback):\n        def on_epoch_end(self, epoch: int, logs: dict[str, Any] | None = None) -> None:\n            epoch_variable.assign(epoch + 1)\n            path = manager.save(checkpoint_number=epoch + 1)\n            (work / \"latest_checkpoint.json\").write_text(json.dumps({\n                \"checkpoint\": str(Path(path).relative_to(work)),\n                \"completed_epoch\": epoch + 1,\n                \"val_loss\": None if logs is None else float(logs.get(\"val_loss\", float(\"nan\"))),\n            }, indent=2))\n            print(f\"[checkpoint] saved epoch {epoch + 1} -> {path}\", flush=True)\n\n    best_path = work / \"best.weights.h5\"\n    callbacks = [\n        CheckpointEveryEpoch(),\n        tf.keras.callbacks.ModelCheckpoint(best_path, monitor=\"val_accuracy\", mode=\"max\", save_best_only=True, save_weights_only=True, verbose=1),\n        tf.keras.callbacks.CSVLogger(str(work / \"training_log.csv\"), append=initial_epoch > 0),\n        tf.keras.callbacks.EarlyStopping(monitor=\"val_accuracy\", mode=\"max\", patience=10, restore_best_weights=True, verbose=1),\n        tf.keras.callbacks.ReduceLROnPlateau(monitor=\"val_loss\", factor=0.5, patience=4, min_lr=1e-5, verbose=1),\n    ]\n    counts = np.bincount(y_train, minlength=len(class_names)).astype(np.float32)\n    weights = np.sqrt(max(float(counts.max()), 1.0) / np.maximum(counts, 1.0))\n    class_weight = {i: float(min(weights[i], 4.0)) for i in range(len(class_names))}\n    print(f\"[train] GPU={tf.config.list_physical_devices('GPU')}; samples train/val/test={len(y_train)}/{len(y_val)}/{len(y_test)}; labels={len(class_names)}\", flush=True)\n    print(f\"[train] input shape={x_train.shape[1:]}; output classes={len(class_names)}\", flush=True)\n    model.fit(\n        x_train, y_train,\n        validation_data=(x_val, y_val),\n        batch_size=args.batch_size,\n        epochs=epochs,\n        initial_epoch=initial_epoch,\n        class_weight=class_weight,\n        shuffle=True,\n        callbacks=callbacks,\n        verbose=2,\n    )\n    if best_path.is_file():\n        model.load_weights(best_path)\n\n    # Static unrolling keeps the exported graph compatible with the app's\n    # builtin-only TFLite interpreter while preserving the training weights.\n    export_model = build_classifier(tf, len(class_names), unroll=True)\n    export_model.set_weights(model.get_weights())\n    model_path = work / \"sign_bridge_wlasl_video.tflite\"\n    converter = tf.lite.TFLiteConverter.from_keras_model(export_model)\n    converter.optimizations = [tf.lite.Optimize.DEFAULT]\n    model_path.write_bytes(converter.convert())\n    interpreter = tf.lite.Interpreter(model_path=str(model_path))\n    interpreter.allocate_tensors()\n    input_detail = interpreter.get_input_details()[0]\n    output_detail = interpreter.get_output_details()[0]\n    sample = np.asarray(x_test[:1], dtype=np.float32)\n    interpreter.set_tensor(input_detail[\"index\"], sample)\n    interpreter.invoke()\n    tflite_output = interpreter.get_tensor(output_detail[\"index\"])\n    keras_output = export_model(sample, training=False).numpy()\n    parity = {\n        \"input_shape\": input_detail[\"shape\"].tolist(),\n        \"output_shape\": output_detail[\"shape\"].tolist(),\n        \"max_absolute_error\": float(np.max(np.abs(keras_output - tflite_output))),\n        \"mean_absolute_error\": float(np.mean(np.abs(keras_output - tflite_output))),\n        \"top1_match\": bool(np.argmax(keras_output[0]) == np.argmax(tflite_output[0])),\n    }\n    if not np.isfinite(tflite_output).all() or parity[\"max_absolute_error\"] > 0.05:\n        raise RuntimeError(f\"TFLite export parity check failed: {parity}\")\n    (work / \"tflite_validation.json\").write_text(json.dumps(parity, indent=2) + \"\\n\")\n    metrics = evaluate_model(model, x_test, y_test, class_names, test_ids, work)\n    (work / \"labels.json\").write_text(json.dumps({\"names\": class_names}, ensure_ascii=False, indent=2) + \"\\n\")\n    (work / \"evaluation.json\").write_text(json.dumps(metrics, indent=2) + \"\\n\")\n    (work / \"model_metadata.json\").write_text(json.dumps({\n        \"model\": model_path.name,\n        \"task\": \"isolated WLASL gloss classification\",\n        \"input\": [1, FRAMES, LANDMARKS, COORDS],\n        \"output\": [1, len(class_names)],\n        \"output_semantics\": \"softmax gloss probabilities\",\n        \"labels_file\": \"labels.json\",\n        \"dataset\": f\"WLASL{NSLT_SUBSET}\",\n        \"classes\": len(class_names),\n        \"test_metrics\": metrics,\n        \"split_policy\": \"Official NSLT split; signer-disjointness is not assumed.\",\n        \"preprocessing\": \"MediaPipe Holistic normalized image landmarks; right hand, left hand, pose indices 11..16, 132 selected face landmarks; per-frame wrist-center/max-absolute hand scale; resampled to 64 frames.\",\n        \"note\": \"Raw WLASL clip recognition is isolated-word recognition, not continuous sentence translation.\",\n    }, indent=2) + \"\\n\")\n    print(f\"[complete] TFLite saved: {model_path} ({model_path.stat().st_size:,} bytes)\", flush=True)\n    print(\"[complete] evaluation:\", json.dumps(metrics, sort_keys=True), flush=True)\n\n\ndef main() -> None:\n    args = args_parser()\n    args.data_root = args.data_root.resolve()\n    args.work_dir.mkdir(parents=True, exist_ok=True)\n    if not (args.data_root / \"videos\").is_dir():\n        candidates = [p.parent for p in Path(\"/kaggle/input\").rglob(\"WLASL_v0.3.json\")]\n        candidates = [p for p in candidates if (p / \"videos\").is_dir()]\n        if len(candidates) == 1:\n            args.data_root = candidates[0]\n        else:\n            mounted = sorted(str(p) for p in Path(\"/kaggle/input\").iterdir()) if Path(\"/kaggle/input\").exists() else []\n            raise FileNotFoundError(\n                f\"Could not locate WLASL videos. configured={args.data_root}, \"\n                f\"candidate_roots={[str(p) for p in candidates]}, mounted_inputs={mounted}\"\n            )\n    if not args.task_model.is_file():\n        candidates = list(Path(\"/kaggle/input\").rglob(TASK_FILENAME))\n        if len(candidates) == 1:\n            args.task_model = candidates[0]\n        else:\n            raise FileNotFoundError(\n                f\"Holistic task model missing at {args.task_model}; \"\n                f\"candidates={[str(p) for p in candidates]}\"\n            )\n\n    rows, class_names, summary = read_records(args.data_root)\n    if args.mode == \"preflight\":\n        rows, class_names = make_preflight_rows(rows, class_names)\n        args.epochs = 1\n        args.extract_chunk_size = 16\n        print(f\"[preflight] using {len(rows)} clips across {len(class_names)} glosses\")\n    summary[\"mode\"] = args.mode\n    summary[\"selected_examples_by_split\"] = dict(Counter(row[\"split\"] for row in rows))\n    summary[\"classes_selected\"] = len(class_names)\n    summary[\"max_epochs\"] = args.epochs\n    summary[\"feature_schema\"] = \"signbridge.android_holistic180.v1\"\n    (args.work_dir / \"dataset_summary.json\").write_text(json.dumps(summary, indent=2) + \"\\n\")\n    with (args.work_dir / \"manifest.csv\").open(\"w\", newline=\"\", encoding=\"utf-8\") as manifest_file:\n        writer = csv.DictWriter(manifest_file, fieldnames=(\n            \"video_id\", \"gloss\", \"split\", \"signer_id\", \"frame_start\", \"frame_end\", \"fps\"))\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({key: row[key] for key in writer.fieldnames})\n    (args.work_dir / \"labels.json\").write_text(json.dumps({\"names\": class_names}, ensure_ascii=False, indent=2) + \"\\n\")\n    print(\"[audit]\", json.dumps({k: v for k, v in summary.items() if k != \"class_names\"}, sort_keys=True), flush=True)\n\n    started = time.time()\n    extract_chunks(rows, class_names, args.task_model, args.work_dir, args.extract_chunk_size, args.mode)\n    (args.work_dir / \"extraction_complete.json\").write_text(json.dumps({\n        \"elapsed_seconds\": time.time() - started,\n        \"chunks\": {split: len(list((args.work_dir / \"features\" / split).glob(\"chunk_*.npz\"))) for split in (\"train\", \"val\", \"test\")},\n    }, indent=2) + \"\\n\")\n    if args.mode == \"preflight\":\n        # Check the complete data/model/export path without spending time on a full run.\n        import tensorflow as tf\n        if not tf.config.list_physical_devices(\"GPU\"):\n            raise RuntimeError(\"Preflight requested GPU but TensorFlow sees no GPU\")\n        x_train, y_train, _, _ = load_split(args.work_dir, \"train\")\n        x_val, y_val, _, _ = load_split(args.work_dir, \"val\")\n        x_test, _, _, _ = load_split(args.work_dir, \"test\")\n        model = build_classifier(tf, len(class_names))\n        model.compile(optimizer=\"adam\", loss=\"sparse_categorical_crossentropy\", metrics=[\"accuracy\"])\n        model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=1, batch_size=4, verbose=2)\n        path = args.work_dir / \"preflight.tflite\"\n        export_model = build_classifier(tf, len(class_names), unroll=True)\n        export_model.set_weights(model.get_weights())\n        path.write_bytes(tf.lite.TFLiteConverter.from_keras_model(export_model).convert())\n        interpreter = tf.lite.Interpreter(model_path=str(path))\n        interpreter.allocate_tensors()\n        input_shape = interpreter.get_input_details()[0][\"shape\"].tolist()\n        output_shape = interpreter.get_output_details()[0][\"shape\"].tolist()\n        if input_shape != [1, FRAMES, LANDMARKS, COORDS] or output_shape != [1, len(class_names)]:\n            raise RuntimeError(f\"Unexpected TFLite contract: {input_shape} -> {output_shape}\")\n        interpreter.set_tensor(interpreter.get_input_details()[0][\"index\"], np.asarray(x_test[:1], dtype=np.float32))\n        interpreter.invoke()\n        smoke_output = interpreter.get_tensor(interpreter.get_output_details()[0][\"index\"])\n        if not np.isfinite(smoke_output).all():\n            raise RuntimeError(\"Preflight TFLite inference produced non-finite values\")\n        print(\"[preflight] sample splits:\", len(x_train), len(x_val), len(x_test), flush=True)\n        print(\"[preflight] TFLite input/output:\", input_shape, output_shape, flush=True)\n        print(\"[preflight] PASS: video decode -> Holistic -> 64x180x3 -> GPU train -> TFLite export\", flush=True)\n        return\n    train(args, rows, class_names, args.work_dir, args.epochs)\n\n\nif __name__ == \"__main__\":\n    main()\n"
trainer_path = Path("/kaggle/working/train_sign_bridge_wlasl_video.py")
trainer_path.parent.mkdir(parents=True, exist_ok=True)
trainer_path.write_text(TRAINER_SOURCE, encoding="utf-8")
subprocess.run(
    [sys.executable, "-u", str(trainer_path), "--mode", RUN_MODE],
    check=True,
)
